In [1]:
import pandas as pd

In [2]:
df_original = pd.read_csv('Children Recode_final.csv')
df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1, inplace=True)
df_original.head()

,Child_age,Mother_education,Age_first_sex,Pregnancy_terminated,Wealth_index,Place_residence,BMI,Children_under5,Total_children_ever_born,Child_sex,...,Ethnicity_2,Ethnicity_3,Ethnicity_4,Ethnicity_5,Ethnicity_6,Ethnicity_7,Ethnicity_8,Ethnicity_9,Ethnicity_10,Malnurished
0,17,1,14,0,1,2,22.00,1,1,2,...,1,0,0,0,0,0,0,0,0,0
1,40,2,17,1,2,2,25.10,2,2,1,...,0,0,0,0,0,0,1,0,0,1
2,59,2,17,1,2,2,25.10,2,2,2,...,0,0,0,0,0,0,1,0,0,1
3,55,2,17,0,2,2,21.53,1,1,2,...,0,0,0,0,0,0,1,0,0,1
4,14,1,16,0,1,2,28.03,1,1,1,...,0,0,0,0,0,0,1,0,0,0


In [3]:
from sklearn.model_selection import train_test_split

X = df_original.drop(columns = ['Malnurished'])
y = df_original['Malnurished']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
from sklearn.preprocessing import MinMaxScaler

columns_to_scale = ['Child_age', 'Age_first_sex', 'BMI', 'Mother_age_current', 'Mother_age_at_first_birth']
scaler = MinMaxScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test_scaled[columns_to_scale] = scaler.transform(X_test[columns_to_scale])


In [5]:
X_train.shape

(1791, 34)

In [6]:

# # For Early Stopping
# import tensorflow as tf

# early_stopping = tf.keras.callbacks.EarlyStopping(
#     monitor = 'val_loss',
#     min_delta = 0.0001,
#     patience = 20,
#     verbose = 1,
#     mode = 'auto',
#     baseline = None,
#     restore_best_weights = False,
# )

In [8]:
from tensorflow import keras

# Define the model properly with an Input layer
model = keras.Sequential([
    keras.Input(shape=(34,)),  
    keras.layers.Dense(25, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train_scaled, y_train, epochs=25)

Epoch 1/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4137 - loss: 0.8887 
Epoch 2/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6540 - loss: 0.6352
Epoch 3/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6590 - loss: 0.6273
Epoch 4/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6500 - loss: 0.6259
Epoch 5/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6715 - loss: 0.6124
Epoch 6/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6630 - loss: 0.6213
Epoch 7/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6654 - loss: 0.6068
Epoch 8/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6818 - loss: 0.6122
Epoch 9/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6367 - loss: 0.6342
Epoch 10/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6569 - loss: 0.6228
Epoch 11/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6555 - loss: 0.6327
Epoch 12/25
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6681 - l

In [9]:
model.evaluate(X_test_scaled, y_test)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6948 - loss: 0.6024  


[0.593620240688324, 0.6875]

In [10]:
yp=model.predict(X_test_scaled)
y_pred = []
for element in yp:
    if element > 0.5:
        y_pred.append(1)
    else:
        y_pred.append(0)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


In [11]:
pd.crosstab(y_test, y_pred)

col_0,0,1
Malnurished,,
0,283,23
1,117,25


In [12]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.71      0.92      0.80       306
           1       0.52      0.18      0.26       142

    accuracy                           0.69       448
   macro avg       0.61      0.55      0.53       448
weighted avg       0.65      0.69      0.63       448

